# Fase 3 · Núcleo algorítmico, eficiencia y POO
## Análisis de licitaciones públicas de ChileCompra · Enero 2025

**MCDI500 · Grupo 2**

Partimos de `F2_Preprocesamiento`. Esa fase exploró la base y aplicó tres transformaciones al conjunto exportado: convirtió fechas, convirtió montos y ordenó el tamaño del proveedor. En esta fase las ejecutamos desde un módulo de Python, medimos dos formas de hacer un control temporal y verificamos que el resultado completo coincida con el de Fase 2.

**Para ejecutar:** mantén este notebook en `F3/`, el módulo en `src/` y el CSV original en `data/raw/04LicitacionesEne.csv`. Abre el notebook en el entorno del repositorio y ejecuta todas las celdas. Se requiere Python con pandas, scikit-learn y Jupyter.

## 0. Verificar el entorno de ejecución

Antes de cargar datos comprobamos que el notebook se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías que usa esta fase tienen las versiones declaradas en `requirements.txt`. Es la misma verificación de la sección 8 de `F1_Definicion`, reducida a lo que necesita la Fase 3.

Importa porque las mediciones de tiempo y memoria de las secciones siguientes dependen del entorno: si cambia la versión de pandas, pueden cambiar las cifras. La celda se detiene si no hay entorno virtual o si falta una librería, y avisa si alguna versión no coincide con la declarada.

In [1]:
# Verificar el entorno: intérprete, entorno virtual y versiones frente a requirements.txt.
import platform
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

# Paquetes que usa la Fase 3, con el nombre que tienen en requirements.txt.
PAQUETES_F3 = ["pandas", "numpy", "scikit-learn"]


def leer_versiones_declaradas(ruta_requirements, paquetes):
    """Devuelve {paquete: versión} de requirements.txt para los paquetes pedidos."""
    declaradas = {}
    for linea in ruta_requirements.read_text(encoding="utf-8").splitlines():
        nombre, _, version = linea.strip().partition("==")
        if nombre.lower() in paquetes:
            declaradas[nombre.lower()] = version
    return declaradas


def verificar_entorno(paquetes):
    """Comprueba el entorno virtual y compara versiones instaladas y declaradas.

    Se detiene si no hay entorno virtual o si falta un paquete. Devuelve una
    tabla con la versión declarada e instalada de cada paquete.
    """
    inicio = Path.cwd().resolve()
    raiz = next((p for p in (inicio, *inicio.parents) if (p / "requirements.txt").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró requirements.txt: abre el notebook dentro del repositorio.")
    if sys.prefix == sys.base_prefix:
        raise RuntimeError("No se usa un entorno virtual: cambia el kernel a 'Python (grupo2-mcdi500)'.")

    declaradas = leer_versiones_declaradas(raiz / "requirements.txt", paquetes)
    filas = []
    for paquete in paquetes:
        try:
            instalada = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            instalada = None
        filas.append({"Paquete": paquete, "Declarada": declaradas.get(paquete),
                      "Instalada": instalada, "Coincide": instalada == declaradas.get(paquete)})
    tabla = pd.DataFrame(filas)

    faltantes = tabla.loc[tabla["Instalada"].isna(), "Paquete"].tolist()
    if faltantes:
        raise ModuleNotFoundError(f"Faltan paquetes: {faltantes}. Ejecuta: pip install -r requirements.txt")

    interprete = Path(sys.executable)
    if interprete.is_relative_to(raiz):
        interprete = interprete.relative_to(raiz)
    print(f"Intérprete: {interprete} | Python {platform.python_version()} | {platform.system()}")
    print("[OK] Entorno virtual activo")
    return tabla


versiones = verificar_entorno(PAQUETES_F3)
display(versiones)
if not versiones["Coincide"].all():
    print("[AVISO] Hay versiones distintas a las declaradas: las cifras de tiempo pueden variar.")
else:
    print(f"[OK] Las {len(versiones)} versiones coinciden con requirements.txt")

Intérprete: .venv/bin/python | Python 3.12.3 | Darwin
[OK] Entorno virtual activo


,Paquete,Declarada,Instalada,Coincide
0,pandas,3.1.0,3.1.0,True
1,numpy,2.5.4,2.5.4,True
2,scikit-learn,1.9.1,1.9.1,True


[OK] Las 3 versiones coinciden con requirements.txt


## 1. Cargar el mismo archivo que en Fase 2

Buscamos la raíz del repositorio a partir de la carpeta donde se ejecuta el notebook. Leemos el CSV con la misma codificación y detección de separador que usó Fase 2. Mostramos dimensiones y monedas para comprobar que la entrada es la esperada. La columna de moneda se conserva, por lo que los montos siguen siendo comparables dentro de su propia unidad.

In [2]:
from pathlib import Path
import sys
import platform
from timeit import repeat

import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

relativa = Path("data/raw/04LicitacionesEne.csv")
inicio = Path.cwd()
raiz = next((p for p in (inicio, *inicio.parents) if (p / relativa).is_file()), None)
if raiz is None:
    raise FileNotFoundError(f"Busca el archivo {relativa} dentro del repositorio.")
sys.path.insert(0, str(raiz))

from src.pipeline_licitaciones import (
    Bitacora, ConversorFechas, ConversorMontos, OrdenadorTamano, Pipeline,
    FECHAS, MONTOS, ORDEN, TAMANO, cargar_licitaciones,
)

datos_originales = cargar_licitaciones(raiz / relativa)
print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"Entrada: {datos_originales.shape[0]:,} filas y {datos_originales.shape[1]} columnas")
display(datos_originales["MonedaOferta"].value_counts(dropna=False).to_frame("Filas"))

Python 3.12.3 | pandas 3.1.0 | scikit-learn 1.9.1
Entrada: 45,741 filas y 74 columnas


,Filas
MonedaOferta,
CLP,45604
CLF,105
USD,23
UTM,9


## 2. Ejecutar las transformaciones como objetos

| Clase | Qué hace | Regla conservada de Fase 2 |
|---|---|---|
| `ConversorFechas` | Convierte publicación, cierre y adjudicación | Formato día-mes-año y hora:minuto. Un valor inválido queda como `NaT`. |
| `ConversorMontos` | Convierte monto neto y total a números | Quita espacios exteriores y cambia coma decimal por punto. Los valores no interpretables quedan nulos. |
| `OrdenadorTamano` | Ordena el tamaño de proveedor | Usa Micro, Pequeña, Mediana, Grande y NoClasificado. Si aparece otra categoría, conserva el tipo original. |

**Encapsulamiento.** Cada conversor guarda su configuración internamente, por ejemplo `_formato` y `_orden`. El método público `transformar()` comprueba que existan las columnas antes de aplicar la regla.

**Herencia.** Las tres clases heredan de `Transformador` y reutilizan la comprobación de columnas. Cada hija define su propia operación en `_aplicar()`.

**Polimorfismo.** `Pipeline` llama a `paso.transformar(datos)` para cada objeto, con independencia del tipo de transformación. Las funciones de carga y medición quedan separadas de las clases.

**Patrón Observer.** Suscribimos la bitácora para que reciba los controles de cada paso. El pipeline puede ejecutar las mismas transformaciones con o sin esa bitácora.

**Decisión sobre recursividad.** El CSV es tabular y las transformaciones siguen una secuencia fija. Usamos división funcional y un bucle porque la cantidad de pasos es conocida y no hay una jerarquía de profundidad variable que recorrer.

In [3]:
bitacora = Bitacora()
pipeline = Pipeline([ConversorFechas(), ConversorMontos(), OrdenadorTamano()])
pipeline.suscribir(bitacora.registrar)

resultado_f3 = pipeline.transformar(datos_originales)
display(bitacora.tabla())
display(resultado_f3[[*FECHAS, *MONTOS, TAMANO]].dtypes.to_frame("Tipo final"))

,Paso,Filas entrada,Filas salida,Columna,Nulos originales,Fallos de conversión,Detalle
0,ConversorFechas,45741,45741,FechaPublicacion,0.0,0.0,NaN
1,ConversorFechas,45741,45741,FechaCierre,0.0,0.0,NaN
2,ConversorFechas,45741,45741,FechaAdjudicacion,91.0,0.0,NaN
3,ConversorMontos,45741,45741,MontoNetoOferta,0.0,0.0,NaN
4,ConversorMontos,45741,45741,MontoTotalOferta,0.0,0.0,NaN
5,OrdenadorTamano,45741,45741,TamanoProveedor,NaN,NaN,Categoría ordenada


,Tipo final
FechaPublicacion,datetime64[us]
FechaCierre,datetime64[us]
FechaAdjudicacion,datetime64[us]
MontoNetoOferta,Float64
MontoTotalOferta,Float64
TamanoProveedor,category


## 3. Comprobar la equivalencia con Fase 2

Reconstruimos la salida de Fase 2 con las instrucciones de conversión del notebook original. La referencia se calcula desde el CSV original en memoria y no llama al módulo de Fase 3. Comparamos los DataFrames completos, incluidos orden, tipos, categorías, valores y nulos. Así evitamos que la lectura de un CSV exportado cambie los tipos antes de comparar.

In [4]:
def ejecutar_referencia_f2(df):
    # Conversión de fechas: mismas reglas del paso 14 de Fase 2.
    df_trabajo = df.copy()
    for columna in ["FechaPublicacion", "FechaCierre", "FechaAdjudicacion"]:
        df_trabajo[columna] = pd.to_datetime(
            df[columna], format="%d-%m-%Y %H:%M", errors="coerce"
        )

    # Conversión de montos: misma regla del paso 18 de Fase 2.
    for columna in ["MontoNetoOferta", "MontoTotalOferta"]:
        original = df_trabajo[columna].astype("string").str.strip()
        texto_convertido = original.str.replace(",", ".", regex=False)
        df_trabajo[columna] = pd.to_numeric(texto_convertido, errors="coerce")

    # Categoría ordenada: misma condición de los pasos 26 y 27 de Fase 2.
    orden_tamano = ["Micro", "Pequeña", "Mediana", "Grande", "NoClasificado"]
    valores_actuales = set(df_trabajo["TamanoProveedor"].dropna().unique())
    if not valores_actuales - set(orden_tamano):
        tipo_tamano = pd.CategoricalDtype(categories=orden_tamano, ordered=True)
        df_trabajo["TamanoProveedor"] = df_trabajo["TamanoProveedor"].astype(tipo_tamano)
    return df_trabajo


resultado_f2 = ejecutar_referencia_f2(datos_originales)
pd.testing.assert_frame_equal(resultado_f2, resultado_f3, check_exact=True)
print(f"Equivalencia comprobada: {len(resultado_f3):,} filas y {resultado_f3.shape[1]} columnas.")

Equivalencia comprobada: 45,741 filas y 74 columnas.


## 4. Medir dos implementaciones del control temporal

Buscamos registros cuyo cierre antecede a la publicación o cuya adjudicación antecede al cierre. Una comparación se evalúa solo cuando sus dos fechas están presentes. Primero comprobamos las dos versiones con casos controlados y luego exigimos que marquen las mismas filas en el conjunto real.

Medimos solo el cálculo de la máscara, sobre todas las filas y con siete repeticiones por versión. Reportamos el menor tiempo observado. Ambas implementaciones recorren las filas y tienen costo temporal **O(n)**. La diferencia medida muestra cuánto cuesta ejecutar las comparaciones en Python frente a realizarlas por columnas con pandas. El tiempo depende del equipo. El control solo reporta señales y no cambia el conjunto.

In [5]:
def control_iterativo(datos):
    marcas = []
    for publicacion, cierre, adjudicacion in datos[list(FECHAS)].itertuples(index=False, name=None):
        cierre_anticipado = pd.notna(publicacion) and pd.notna(cierre) and cierre < publicacion
        adjudicacion_anticipada = pd.notna(cierre) and pd.notna(adjudicacion) and adjudicacion < cierre
        marcas.append(cierre_anticipado or adjudicacion_anticipada)
    return pd.Series(marcas, index=datos.index, dtype=bool)


def control_vectorizado(datos):
    publicacion, cierre, adjudicacion = (datos[c] for c in FECHAS)
    return (cierre < publicacion) | (adjudicacion < cierre)


casos = pd.DataFrame([
    ["2025-01-01", "2025-01-03", "2025-01-05"],
    ["2025-01-03", "2025-01-02", None],
    [None, "2025-01-05", "2025-01-04"],
    ["2025-01-01", None, "2025-01-05"],
], columns=FECHAS).apply(pd.to_datetime)
esperado = pd.Series([False, True, True, False])
pd.testing.assert_series_equal(control_iterativo(casos), esperado)
pd.testing.assert_series_equal(control_vectorizado(casos), esperado)

mascara = control_vectorizado(resultado_f3)
pd.testing.assert_series_equal(control_iterativo(resultado_f3), mascara)

mediciones = []
for nombre, operacion in [("Iterativa", control_iterativo), ("Vectorizada", control_vectorizado)]:
    tiempos = repeat(lambda: operacion(resultado_f3), number=1, repeat=7)
    mediciones.append({"Implementación": nombre, "Filas": len(resultado_f3),
                       "Repeticiones": len(tiempos), "Mínimo (ms)": min(tiempos) * 1000})

tabla_tiempos = pd.DataFrame(mediciones)
display(tabla_tiempos.round(3))
razon = tabla_tiempos.loc[0, "Mínimo (ms)"] / tabla_tiempos.loc[1, "Mínimo (ms)"]
print(f"Razón de tiempos iterativa/vectorizada: {razon:.2f}×")
print(f"Filas con alerta temporal: {mascara.sum():,}. Las dos versiones identificaron las mismas filas.")

,Implementación,Filas,Repeticiones,Mínimo (ms)
0,Iterativa,45741,7,82.193
1,Vectorizada,45741,7,0.145


Razón de tiempos iterativa/vectorizada: 566.85×
Filas con alerta temporal: 0. Las dos versiones identificaron las mismas filas.


### 4.1 Verificar la herramienta de medición

Las mediciones de esta fase usan el módulo `src/medicion.py`. Sus funciones son `medir_tiempo()`, que guarda el menor de varios tiempos con `timeit`, `medir_memoria()`, que registra el pico de memoria con `tracemalloc`, y `comparar()`, que mide varias versiones y exige que todas entreguen el mismo resultado. Antes de usarlo sobre los datos comprobamos que funciona en tres escenarios:

- **Caso normal.** Dos formas de sumar los números de 1 a n deben dar el mismo resultado y una fila por versión.
- **Caso límite.** Una sola repetición es válida, y medir la memoria no debe alterar el resultado de la función medida.
- **Excepciones.** Dos versiones con resultados distintos, una sola versión o cero repeticiones deben detenerse con un error claro. Si alguno de esos casos no fallara, la celda se detiene.

In [6]:
# Verificar src/medicion.py en casos normal, límite y de excepción.
from src.medicion import comparar, medir_memoria, medir_tiempo

# Caso normal: dos formas de sumar 1..n dan lo mismo y se obtiene una fila por versión.
n = 10_000
prueba = comparar({"Bucle": lambda: sum(range(n + 1)),
                   "Fórmula": lambda: n * (n + 1) // 2}, repeticiones=3)
assert list(prueba["Implementación"]) == ["Bucle", "Fórmula"]
assert (prueba[["Tiempo mínimo (ms)", "Memoria pico (MB)"]] >= 0).all().all()
print("[OK] Caso normal: dos versiones equivalentes, una fila por versión")

# Caso límite: una sola repetición es válida y medir la memoria no altera el resultado.
assert medir_tiempo(lambda: None, repeticiones=1) >= 0
assert medir_memoria(lambda: 42)[0] == 42
print("[OK] Caso límite: una repetición y resultado intacto")

# Excepciones: cada caso debe fallar con el error indicado.
casos_error = {
    "Resultados distintos": (lambda: comparar({"a": lambda: 1, "b": lambda: 2}), AssertionError),
    "Una sola versión": (lambda: comparar({"a": lambda: 1}), ValueError),
    "Cero repeticiones": (lambda: medir_tiempo(lambda: 1, repeticiones=0), ValueError),
}
for caso, (llamada, error_esperado) in casos_error.items():
    try:
        llamada()
    except error_esperado as error:
        print(f"[OK] Excepción esperada · {caso}: {type(error).__name__} ({error})")
    else:
        raise AssertionError(f"{caso}: se esperaba {error_esperado.__name__} y no hubo error.")

[OK] Caso normal: dos versiones equivalentes, una fila por versión
[OK] Caso límite: una repetición y resultado intacto
[OK] Excepción esperada · Resultados distintos: AssertionError (b entrega un resultado distinto al de la referencia.)
[OK] Excepción esperada · Una sola versión: ValueError (Se necesitan al menos dos implementaciones para comparar.)
[OK] Excepción esperada · Cero repeticiones: ValueError (Se necesita al menos una repetición.)


### 4.2 Tiempo y memoria del control temporal

La sección 4 midió el tiempo de las dos versiones del control temporal. Ahora repetimos la comparación con `comparar()` para agregar la memoria pico, porque una versión más rápida puede costar más memoria y la decisión debe considerar ambas dimensiones. La memoria se mide en una ejecución separada de la de tiempo, porque `tracemalloc` hace más lento el código que registra. `comparar()` vuelve a exigir que las dos versiones marquen exactamente las mismas filas.

In [7]:
# Medir tiempo y memoria de las dos versiones del control temporal de la sección 4.
comparacion_control = comparar({
    "Iterativa": lambda: control_iterativo(resultado_f3),
    "Vectorizada": lambda: control_vectorizado(resultado_f3),
})
display(comparacion_control)

cifras = comparacion_control.set_index("Implementación")
razon_tiempo = cifras.loc["Iterativa", "Veces el tiempo de la más rápida"]
razon_memoria = cifras.loc["Iterativa", "Memoria pico (MB)"] / cifras.loc["Vectorizada", "Memoria pico (MB)"]
print(f"Sobre {len(resultado_f3):,} filas, la versión iterativa tarda {razon_tiempo:,.0f} veces más "
      f"y usa {razon_memoria:,.0f} veces más memoria pico que la vectorizada.")

,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,Iterativa,81.613,4.002,553.311,7
1,Vectorizada,0.147,0.224,1.000,7


Sobre 45,741 filas, la versión iterativa tarda 553 veces más y usa 18 veces más memoria pico que la vectorizada.


**Interpretación.** La versión vectorizada es mejor en las dos dimensiones, así que no hay que elegir entre tiempo y memoria. Las dos versiones tienen complejidad temporal y espacial **O(n)**: ambas recorren cada fila una vez y guardan una marca por fila. La diferencia está en el costo de cada paso.

- **Tiempo.** La versión iterativa evalúa las comparaciones en Python, fila por fila. La vectorizada compara columnas completas en el código compilado de pandas.
- **Memoria.** La versión iterativa crea una tupla por fila con `itertuples` y acumula las marcas en una lista de objetos de Python antes de convertirla en serie. La vectorizada guarda cada marca en un arreglo booleano de un byte por fila.

**Decisión.** Los controles por fila de la Fase 4 se escribirán en forma vectorizada. La versión iterativa se conserva solo como referencia para verificar el resultado. Las cifras exactas dependen del equipo; lo que se mantiene entre equipos es la proporción entre las dos versiones.

### 4.3 Leer el archivo: tres estrategias

Cada fase empieza leyendo el CSV original, de unos 60 MB. El cargador actual, `cargar_licitaciones()`, usa las mismas opciones que la Fase 2: deja que pandas detecte el separador (`sep=None`), lo que obliga a usar el motor de lectura escrito en Python. Comparamos tres estrategias:

- **A. Cargador actual.** Detecta el separador con el motor Python.
- **B. Separador explícito.** Declara `sep=";"`, que ya conocemos desde la Fase 1, y usa el motor C de pandas.
- **C. Separador explícito y `usecols`.** Como B, pero lee solo las 8 columnas que usa esta fase.

Hacemos dos comparaciones porque C entrega menos columnas. En la primera, B debe entregar exactamente el mismo DataFrame completo que A. En la segunda, C debe coincidir con esas 8 columnas leídas por B. Cada lectura se repite tres veces y se conserva el menor tiempo, porque la estrategia A tarda más de un segundo por lectura.

In [8]:
# Comparar tres formas de leer el CSV original: tiempo, memoria y resultado idéntico.
ruta_csv = raiz / relativa
COLUMNAS_F3 = ["NroLicitacion", "MonedaOferta", *FECHAS, *MONTOS, TAMANO]

# Comparación 1: archivo completo. B debe entregar el mismo DataFrame que el cargador actual.
lectura_completa = comparar({
    "A. Cargador actual (sep=None, motor Python)": lambda: cargar_licitaciones(ruta_csv),
    "B. Separador explícito (sep=';', motor C)": lambda: pd.read_csv(ruta_csv, sep=";", encoding="latin-1"),
}, repeticiones=3)
display(lectura_completa)

# Comparación 2: solo las columnas de esta fase. C debe coincidir con esas columnas de B.
lectura_columnas = comparar({
    "B. Separador explícito, 74 columnas": lambda: pd.read_csv(ruta_csv, sep=";", encoding="latin-1")[COLUMNAS_F3],
    "C. Separador explícito y usecols, 8 columnas": lambda: pd.read_csv(
        ruta_csv, sep=";", encoding="latin-1", usecols=COLUMNAS_F3)[COLUMNAS_F3],
}, repeticiones=3)
display(lectura_columnas)

for nombre, tabla in [("B frente a A", lectura_completa), ("C frente a B", lectura_columnas)]:
    veces_tiempo = tabla["Veces el tiempo de la más rápida"].iloc[0]
    veces_memoria = tabla["Memoria pico (MB)"].iloc[0] / tabla["Memoria pico (MB)"].iloc[1]
    print(f"{nombre}: {veces_tiempo:.1f} veces más rápida y {veces_memoria:.1f} veces menos memoria pico, "
          "con el mismo resultado.")

,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,"A. Cargador actual (sep=None, motor Python)",1332.268,229.291,4.211,3
1,"B. Separador explícito (sep=';', motor C)",316.362,56.278,1.000,3


,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,"B. Separador explícito, 74 columnas",316.973,56.27,3.075,3
1,"C. Separador explícito y usecols, 8 columnas",103.079,8.81,1.000,3


B frente a A: 4.2 veces más rápida y 4.1 veces menos memoria pico, con el mismo resultado.
C frente a B: 3.1 veces más rápida y 6.4 veces menos memoria pico, con el mismo resultado.


**Interpretación.**

- **A frente a B.** Entregan exactamente el mismo DataFrame: 45.741 filas y 74 columnas, con los mismos tipos y valores. La diferencia está solo en cómo se lee. Al detectar el separador, pandas usa el motor Python, que procesa el archivo línea por línea y arma listas intermedias. Con el separador declarado, el motor C lee directamente. Como el separador es conocido, detectarlo no aporta nada y cuesta tiempo y memoria.
- **B frente a C.** `usecols` descarta las columnas durante la lectura y no después. El tiempo y la memoria bajan de forma marcada, aunque no en proporción exacta a las columnas (8 de 74), porque el archivo se sigue recorriendo completo.
- **Complejidad.** Las tres estrategias son **O(n · m)** en tiempo, con n filas y m columnas leídas: todas recorren el archivo una vez. B mejora la constante de cada paso y C reduce m. En memoria, B y C guardan solo el resultado, mientras que A suma las estructuras intermedias del motor Python.

**Decisión.**

1. El cargador del proyecto adoptará la estrategia B: entrega los mismos datos con menos tiempo y menos memoria, sin ninguna contrapartida.
2. Esta fase sigue leyendo las 74 columnas, porque exporta el conjunto completo para la Fase 4. `usecols` queda como práctica recomendada para los análisis de la Fase 4 que necesiten pocas columnas.

## 5. Preparar los archivos para Fase 4

La verificación anterior usa todo el conjunto. Ahora creamos entrenamiento y prueba con una semilla fija y una proporción objetivo de 80/20 **por licitación**, de modo que las ofertas de una misma licitación queden juntas. El porcentaje por filas puede diferir. Como las conversiones usan reglas fijas, ningún paso estima medianas, escalas u otros parámetros a partir de los datos de prueba.

El diccionario documenta columnas, tipos y origen. El archivo de parámetros registra las reglas fijas y la semilla. Los CSV se guardan en `data/processed/` con punto y coma y UTF-8 con BOM. Al volver a leerlos, usa el diccionario para reconstruir fechas y categorías cuando haga falta.

In [9]:
semilla = 42
fraccion_prueba = 0.20
grupos = resultado_f3["NroLicitacion"]
if grupos.isna().any():
    raise ValueError("Hay licitaciones sin identificador. Revísalas antes de dividir por licitación.")

divisor = GroupShuffleSplit(n_splits=1, test_size=fraccion_prueba, random_state=semilla)
indices_ent, indices_pru = next(divisor.split(resultado_f3, groups=grupos))
entrenamiento = resultado_f3.iloc[indices_ent]
prueba = resultado_f3.iloc[indices_pru]

assert set(entrenamiento["NroLicitacion"]).isdisjoint(prueba["NroLicitacion"])
assert len(entrenamiento) + len(prueba) == len(resultado_f3)
pd.testing.assert_series_equal(entrenamiento.dtypes, prueba.dtypes)

reglas = {**dict.fromkeys(FECHAS, "Conversión de fecha y hora"),
          **dict.fromkeys(MONTOS, "Conversión numérica con coma decimal"),
          TAMANO: "Orden lógico de tamaño de proveedor"}
diccionario = pd.DataFrame([
    {"Columna": c, "Tipo": str(resultado_f3[c].dtype),
     "Origen": "04LicitacionesEne.csv", "Transformación": reglas.get(c, "Se conserva")}
    for c in resultado_f3.columns
])
parametros = pd.DataFrame([
    ("ConversorFechas", "formato", "%d-%m-%Y %H:%M", "Fijo"),
    ("ConversorMontos", "regla", "Recortar espacios y sustituir coma por punto", "Fijo"),
    ("OrdenadorTamano", "orden", " | ".join(ORDEN), "Fijo"),
    ("División", "grupo", "NroLicitacion", "Fijo"),
    ("División", "semilla", str(semilla), "Fijo"),
    ("División", "fracción de prueba por grupos", str(fraccion_prueba), "Fijo"),
], columns=["Paso", "Parámetro", "Valor", "Origen"])

destino = raiz / "data" / "processed"
destino.mkdir(parents=True, exist_ok=True)
archivos = {"entrenamiento_f3.csv": entrenamiento, "prueba_f3.csv": prueba,
            "diccionario_f3.csv": diccionario, "parametros_f3.csv": parametros}
for nombre, tabla in archivos.items():
    tabla.to_csv(destino / nombre, sep=";", encoding="utf-8-sig", index=False)
    print(f"data/processed/{nombre} · {len(tabla):,} filas")

print(f"Licitaciones únicas: {entrenamiento['NroLicitacion'].nunique():,} en entrenamiento y "
      f"{prueba['NroLicitacion'].nunique():,} en prueba. No hay grupos compartidos.")
display(parametros)

data/processed/entrenamiento_f3.csv · 36,438 filas


data/processed/prueba_f3.csv · 9,303 filas
data/processed/diccionario_f3.csv · 74 filas
data/processed/parametros_f3.csv · 6 filas
Licitaciones únicas: 1,833 en entrenamiento y 459 en prueba. No hay grupos compartidos.


,Paso,Parámetro,Valor,Origen
0,ConversorFechas,formato,%d-%m-%Y %H:%M,Fijo
1,ConversorMontos,regla,Recortar espacios y sustituir coma por punto,Fijo
2,OrdenadorTamano,orden,Micro | Pequeña | Mediana | Grande | NoClasifi...,Fijo
3,División,grupo,NroLicitacion,Fijo
4,División,semilla,42,Fijo
5,División,fracción de prueba por grupos,0.2,Fijo


## 6. Qué llevar al informe técnico

- **Diseño:** enlazar `src/pipeline_licitaciones.py` y explicar qué hace cada clase y cómo se ejecuta el pipeline. Usar la bitácora como evidencia del patrón Observer.
- **Equivalencia:** incluir el resultado de `assert_frame_equal` y aclarar que los controles de Fase 2, como los atípicos, no se usaron para eliminar filas.
- **Eficiencia:** incorporar la tabla de tiempos, el tamaño del conjunto y la explicación de por qué ambas implementaciones tienen complejidad lineal.
- **Trazabilidad:** enlazar el commit que incorpore el notebook y el módulo. Ejecutar todas las celdas en el entorno del repositorio antes de la entrega.